# Run preprocessing (CPU, cluster)

Runs the CPU preprocessing tier end-to-end: builds the cohort (both anchors), extracts ICD
timing, builds non-text covariates, renders the data-availability report, then tokenizes notes.
Ends with the `gsutil` command to ship token batches to GCP for embedding generation
(see [generate_embeddings_gcp.ipynb](generate_embeddings_gcp.ipynb)).

In [ ]:
from __future__ import annotations

import subprocess
import sys
from pathlib import Path


def find_v2_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "config.py").is_file() and (candidate / "pipelines").is_dir():
            return candidate
    raise RuntimeError(f"Could not find v2 root from {start}")


V2_ROOT = find_v2_root()
print(f"v2 root: {V2_ROOT}")
print(f"Python:  {sys.executable}")

## Run stages

Each stage is invoked as a subprocess so a failure on one stage does not prevent inspecting
what already ran. `report_data_availability` renders the combination table inline (it prints
its summary JSON and writes the marginals/combinations/pairwise CSVs to `SURV_PATH`).

In [ ]:
STAGES = [
    ["pipelines.preprocessing.build_cohort"],
    ["pipelines.preprocessing.extract_ICD_times"],
    ["pipelines.preprocessing.generate_all_non_text_covariates"],
    ["pipelines.preprocessing.report_data_availability"],
    ["pipelines.preprocessing.text_preprocessing_and_tokenization"],
]


def run_stage(args: list[str]) -> bool:
    print("\n=== " + " ".join(args) + " ===", flush=True)
    proc = subprocess.run([sys.executable, "-m", *args], cwd=V2_ROOT)
    return proc.returncode == 0


results = {"ok": [], "failed": []}
for args in STAGES:
    if run_stage(args):
        results["ok"].append(args[0])
    else:
        results["failed"].append(args[0])
        print(f"\n[FAIL] {args[0]}", flush=True)

n_ok = len(results["ok"])
n_failed = len(results["failed"])
print(f"\nDone. {n_ok} succeeded, {n_failed} failed.")
if results["failed"]:
    print("Failures:", results["failed"])

## Ship token batches to GCP

Copy the tokenized note batches up to the GCS bucket used by
[generate_embeddings_gcp.ipynb](generate_embeddings_gcp.ipynb). Adjust the bucket path for your
GCP project before running.

In [ ]:
GCS_BUCKET = "gs://<your-bucket>/clinical_text_embedding_project/tokens"

gsutil_cmd = f"gsutil -m rsync -r $BATCHED_DATA_PATH/batched_tokens {GCS_BUCKET}"
print("Run on the cluster shell (not from this notebook) once token batches are written:\n")
print(f"  {gsutil_cmd}")